# 03 - Phase mapping: Si, SiO2, WS2 and a C+Pt cap

Your sample contains four phases and this notebook separates them, twice
over - once from EELS and once from EDX - and then compares the two answers.

> **This is a template. It needs your measurement and will not run without it.**
> Every cell is written and checked, but the file paths in section 1 are
> placeholders. Point them at your data and the notebook runs top to bottom.
> The only exception is the bonus section at the end, which works with the
> workshop's own Si/SiO2 reference spectra.

## Which signal reaches which phase

| Phase | EELS | EDX |
| --- | --- | --- |
| Si | Si-L2,3 at 99 eV, **no** oxygen | Si-Ka 1.740 keV |
| SiO2 | Si-L2,3 **and** O-K at 532 eV | Si-Ka + O-Ka 0.525 keV |
| WS2 | **S-L2,3 at 165 eV** | W-La 8.398 keV + S-Ka 2.307 keV |
| C+Pt | **C-K at 284 eV** | Pt-La 9.442 keV, C-Ka 0.277 keV |

Two entries in that table deserve an explanation, because they are the reason
this notebook is built the way it is.

**EELS reaches WS2 through sulphur, not tungsten.** The lowest tungsten edge
in eXSpy's database is W-M5 at 1809 eV, and platinum starts at Pt-M5 = 2122 eV.
In an 80-600 eV window neither element exists as far as the software is
concerned - `add_elements(["W", "Pt"])` would drop them without a word. So EELS
identifies WS2 by its sulphur and the cap by its carbon.

**EDX cannot separate Si from W on the M line.** Si-Ka sits at 1.740 keV and
W-Ma at 1.776 keV - 36 eV apart, against a detector resolution of roughly
130 eV. They are one peak. Silicon quantified next to WS2 without tungsten in
the model comes out far too high. The way round it is the tungsten L line at
8.398 keV, which is clean.

So the two methods fail in different places, which is exactly why doing both
is worth the beam time. Section 6 compares them.

In [ ]:
# Interactive plots. If they stay blank, use %matplotlib inline instead
# and restart the kernel.
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import hyperspy.api as hs
import exspy  # registers the EELS/EDX signal types

from workshop_data import load_path, list_files, check_elements, check_background_window, energy_range
from phase_analysis import (
    check_xray_lines,
    normalise,
    phase_scores,
    classify,
    phase_fractions,
    plot_phase_map,
    PHASE_COLOURS,
)

print("HyperSpy", hs.__version__, "| exspy", exspy.__version__)

## 1. Point the notebook at your data

Copy your files into `data/`, ideally into a folder of their own. The next
cell lists what is there.

In [ ]:
_ = list_files("*")

**Edit this block.** The three EELS entries and the EDX entry are what the rest
of the notebook runs on. Set `EDX_SI` to `None` if you only have EELS, or
`EELS_HL` to `None` if you only have EDX - the corresponding sections then
skip themselves instead of failing.

In [ ]:
# --- EDIT THIS BLOCK -------------------------------------------------------
EELS_HL  = "my_sample/EELS SI core-loss.dm4"   # 80-600 eV, or None
EELS_LL  = "my_sample/EELS SI low-loss.dm4"    # for zero-loss alignment, or None
EDX_SI   = "my_sample/EDS SI.dm4"              # or None
SURVEY   = "my_sample/ADF Image.dm4"           # or None

BINNING  = [2, 2, 1]      # [x, y, energy]; raise x/y if the fits take too long
BEAM_KV  = 200.0          # accelerating voltage, needed for the EDX line check
# ---------------------------------------------------------------------------

# Elements each method can actually work with. Note what is NOT here:
# W and Pt are missing from the EELS list on purpose - see the header.
EELS_ELEMENTS = ["Si", "O", "S", "C"]
EDX_ELEMENTS  = ["Si", "O", "S", "C", "W", "Pt"]

In [ ]:
# Bound up front so that a missing file leaves one clear message instead of a
# long tail of NameErrors in every cell below.
eels = ll = edx = None

try:
    eels = load_path(EELS_HL, signal_type="EELS") if EELS_HL else None
    ll   = load_path(EELS_LL, signal_type="EELS") if EELS_LL else None
    edx  = load_path(EDX_SI, signal_type="EDS_TEM") if EDX_SI else None
except FileNotFoundError as error:
    print("=" * 72)
    print("NO DATA LOADED - this notebook is a template.")
    print("Edit the block above so the paths point at your own measurement.")
    print("Every section below will skip itself until then.")
    print("=" * 72)
    print(error)

if eels is not None:
    lo, hi = energy_range(eels)
    print(f"EELS  {eels.data.shape}   {lo:.0f} - {hi:.0f} eV")
if edx is not None:
    lo, hi = energy_range(edx)
    print(f"EDX   {edx.data.shape}   {lo:.2f} - {hi:.2f} keV")
print(f"low-loss: {'loaded' if ll is not None else 'not used'}")

In [ ]:
if SURVEY:
    load_path(SURVEY).plot()

## 2. What can each method actually see?

Run this **before** any fitting. It is a two-minute check that prevents the
two most expensive mistakes in this notebook: modelling an element that has no
edge in your window, and quantifying a line that sits underneath another one.

Neither mistake produces an error message. Both produce numbers.

In [ ]:
if eels is not None:
    result = check_elements(eels, EELS_ELEMENTS + ["W", "Pt"])
    print()
    print("W and Pt are listed here only to make the point - they are not in")
    print("EELS_ELEMENTS, because at 80-600 eV eXSpy has no edge for them.")

In [ ]:
lines = check_xray_lines(EDX_ELEMENTS, beam_energy_kv=BEAM_KV, resolution_ev=130.0)

Look at the overlap list. `Si-Ka` and `W-Ma` are 36 eV apart - that is the pair
that decides whether your silicon map is trustworthy. The good news is that
eXSpy's `add_lines()` already picks `W_La` and `Pt_La` at 200 kV rather than the
M lines. The bad news is that it is easy to override that by accident, and
exercise 2 makes you do exactly that so you can see the size of the error.

## 3. EELS: elemental maps

### Why the background is handled differently here than in notebook 01

Notebook 01 strips one power-law background and then models a narrow window.
That works when all the edges sit close together. Here they are spread from
99 eV to 532 eV, and a single power law fitted at 80-95 eV does not describe
the background five hundred electronvolts later.

So this notebook keeps the background **inside** the model
(`auto_background=True`) and fits with `kind="smart"`, which re-fits the
background in front of each edge before fitting that edge. That is what "smart"
means, and this is the situation it was built for.

In [ ]:
if ll is not None:
    ll.align_zero_loss_peak(also_align=[eels], signal_range=(-10.0, 10.0))
    print("aligned on the zero-loss peak")
elif eels is not None:
    print("no low-loss data - skipping alignment.")
    print("Your edges may sit at slightly wrong energies; the maps still work,")
    print("but do not read absolute onsets off this data.")

In [ ]:
if eels is not None:
    eels.add_elements(EELS_ELEMENTS)
    eels_binned = eels.rebin(scale=BINNING)

    m_eels = eels_binned.create_model(auto_background=True)
    m_eels.components

### Inspect the model - interactively **or** in code

Variant A opens sliders, variant B prints the same information as text.

**Variant A is off by default, on purpose.** An open `m.gui()` widget binds every
parameter to a slider, and a fit cannot run while that binding is live - it aborts
with `TraitError: Broken link`. Variant B has no such problem and is what runs when
you execute all cells.

In [ ]:
# --- Variant A: interactive ---
#
# CAUTION: close this widget again before fitting.
#
# m.gui() binds every model parameter to a slider. While a fit runs, the optimiser
# writes each parameter thousands of times per second; every value travels to the
# browser and comes back with float32 precision, and the binding then stops the fit
# with
#
#   TraitError: Broken link ...: the source value changed while updating the target
#
# If that happens to you, re-run the cell above that creates the model - a fresh
# model carries no widget - and fit again. No kernel restart needed.

SHOW_GUI = False

if SHOW_GUI and eels is not None:
    m_eels.gui()
else:
    print("interactive widget off - set SHOW_GUI = True to open it,")
    print("and re-create the model before fitting.")

In [ ]:
# --- Variant B: the same information as text ---
if eels is not None:
    for component in m_eels:
        print(f"{component.name:12s} active={component.active}")

In [ ]:
if eels is not None:
    # kind="smart": background re-fitted before each edge. Slower, but the only
    # sensible choice across a window this wide.
    m_eels.multifit(kind="smart")

In [ ]:
if eels is not None:
    m_eels.plot_results()

Now pull one map per element out of the model. `EELSCLEdge.intensity` is the
fitted edge height, and that is what scales with how much of the element sits
in the pixel.

Silicon has three edges (L3, L2, L1) and sulphur two (L2,3, L1). We take the
strongest of each - L3 for Si, L2,3 for S - rather than adding them, because
the weaker ones are fitted much less reliably.

In [ ]:
if eels is not None:
    wanted = {"Si": "Si_L3", "O": "O_K", "S": "S_L2,3", "C": "C_K"}

    eels_maps = {}
    for element, component_name in wanted.items():
        match = [c for c in m_eels if c.name == component_name]
        if not match:
            print(f"  [missing] no component named {component_name!r} - "
                  f"is {element} really in your energy window?")
            continue
        eels_maps[element] = np.asarray(match[0].intensity.map["values"], float)
        print(f"  [ok] {element:3s} from {component_name}")

    eels_maps = {k: normalise(v) for k, v in eels_maps.items()}

In [ ]:
if eels is not None and eels_maps:
    fig, axes = plt.subplots(1, len(eels_maps), figsize=(3.4 * len(eels_maps), 3.4))
    for ax, (element, data) in zip(np.atleast_1d(axes), eels_maps.items()):
        im = ax.imshow(data, cmap="magma", vmin=0, vmax=1)
        ax.set_title(f"{element} (EELS, normalised)")
        ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=axes, shrink=0.8)
    plt.show()

## 4. EELS: from elements to phases

The rules are written out in `phase_analysis.phase_scores` and they are short
enough to argue with:

* **WS2** - sulphur is present. Nothing else in this sample contains sulphur.
* **C+Pt** - carbon is present. Nothing else contains carbon.
* **SiO2** - silicon *and* oxygen. `min(Si, O)` demands both.
* **Si** - silicon *without* oxygen. The `(1 - O)` factor kills the pixel as
  soon as oxygen shows up.

Each pixel is given to its highest-scoring phase. Pixels whose best score stays
under `threshold` are left unassigned - vacuum, the specimen edge, and noise
should land there.

In [ ]:
if eels is not None and len(eels_maps) == 4:
    THRESHOLD = 0.15   # raise it to shrink the phases, lower it to grow them

    eels_scores = phase_scores(eels_maps)
    eels_labels, phase_names = classify(eels_scores, threshold=THRESHOLD)

    fig, ax = plt.subplots(figsize=(6.5, 5))
    plot_phase_map(eels_labels, phase_names, ax=ax, title="Phase map from EELS")
    plt.show()

    eels_fractions = phase_fractions(eels_labels, phase_names)
elif eels is not None:
    print(f"Need all four maps, have: {sorted(eels_maps)}")

## 5. EDX: the same four phases, independently

Two things differ from notebook 02. First, the element list contains W and Pt,
which EELS could not reach. Second - and this is the part worth slowing down for -
the model does something you need to know about.

### The silicon/tungsten degeneracy, precisely

At 200 kV `add_lines()` picks `W_La` (8.398 keV) and `Pt_La` (9.442 keV), which is
the right choice: those lines are clean. But `create_model()` then builds
components for the **whole** X-ray family of each element, M lines included. So the
model contains, among others:

```
Si_Ka   centre 1.740 keV   amplitude free
W_Ma    centre 1.776 keV   amplitude free      <- 36 eV away
```

Within one family the satellites are tied to the family head - `Si_Kb` follows
`Si_Ka`, `W_Mb` follows `W_Ma`, `W_Lb1` follows `W_La`. But **the M family is not
tied to the L family**: `W_Ma` has its own free amplitude, and the clean L line at
8.4 keV does not constrain it at all.

That leaves two free amplitudes 36 eV apart under a peak roughly 130 eV wide. The
fit has no unique answer there. Whatever it reports as silicon in a tungsten-rich
pixel is arbitrary within a wide range.

The next cell shows the degeneracy in your own model, and the one after it fixes it.

In [ ]:
if edx is not None:
    edx_binned = edx.rebin(scale=BINNING)
    edx_binned.set_elements(EDX_ELEMENTS)
    edx_binned.add_lines()
    print("primary lines chosen:", list(edx_binned.metadata.Sample.xray_lines))

    m_edx = edx_binned.create_model()

    # Which amplitudes is the fit actually free to move, and where do they sit?
    free = [(c.name, float(c.centre.value)) for c in m_edx
            if hasattr(c, "A") and c.A.free and c.active]
    free.sort(key=lambda row: row[1])

    print("\nfree amplitudes in the model:")
    for name, centre in free:
        print(f"  {name:10s} {centre:7.3f} keV")

    print("\npairs of FREE components closer than the detector resolution (130 eV):")
    trouble = []
    for i in range(len(free) - 1):
        gap = (free[i + 1][1] - free[i][1]) * 1000
        if gap < 130:
            trouble.append((free[i][0], free[i + 1][0], gap))
            print(f"  [DEGENERATE] {free[i][0]} and {free[i+1][0]}: {gap:.0f} eV apart")
    if not trouble:
        print("  none - the fit is well posed")

In [ ]:
# --- Fixing the degeneracy -------------------------------------------------
#
# Do NOT simply switch the M components off. The sample really does emit W-M
# X-rays; if the model has nowhere to put them they end up inside Si_Ka and the
# silicon comes out even more wrong.
#
# The honest fix is to tie the M family to the L family with a fixed ratio, so
# only one tungsten amplitude stays free. Calibrate the ratio where there is no
# silicon to confuse it - a pure WS2 region - and then apply it everywhere.
#
# Set M_OVER_L to None to skip this and see the unconstrained result instead.

M_OVER_L = {"W": 0.45, "Pt": 0.45}   # placeholder values - calibrate them, see exercise 3

if edx is not None and M_OVER_L:
    for element, ratio in M_OVER_L.items():
        head_m, head_l = f"{element}_Ma", f"{element}_La"
        try:
            m_edx[head_m].A.twin = m_edx[head_l].A
            m_edx[head_m].A.twin_function_expr = f"{ratio} * x"
            print(f"  {head_m} tied to {head_l} with ratio {ratio}")
        except (KeyError, ValueError) as error:
            print(f"  could not tie {head_m}: {error}")

    still_free = [(c.name, float(c.centre.value)) for c in m_edx
                  if hasattr(c, "A") and c.A.free and c.active]
    still_free.sort(key=lambda row: row[1])
    close = [(a, b, (bc - ac) * 1000)
             for (a, ac), (b, bc) in zip(still_free, still_free[1:])
             if (bc - ac) * 1000 < 130]
    print("\nremaining degenerate pairs:", close if close else "none")

In [ ]:
if edx is not None:
    edx_sum = edx_binned.sum()
    edx_sum.plot(True)   # True = draw the line markers

Zoom the plot above into 1.6-2.2 keV. That is where Si-Ka, W-Ma, W-Mb and
Pt-Ma all live inside a span narrower than three detector resolutions. Seeing
it once is worth more than reading about it.

In [ ]:
# --- Variant A: interactive ---
#
# CAUTION: close this widget again before fitting.
#
# m.gui() binds every model parameter to a slider. While a fit runs, the optimiser
# writes each parameter thousands of times per second; every value travels to the
# browser and comes back with float32 precision, and the binding then stops the fit
# with
#
#   TraitError: Broken link ...: the source value changed while updating the target
#
# If that happens to you, re-run the cell above that creates the model - a fresh
# model carries no widget - and fit again. No kernel restart needed.

SHOW_GUI = False

if SHOW_GUI and edx is not None:
    m_edx.gui()
else:
    print("interactive widget off - set SHOW_GUI = True to open it,")
    print("and re-create the model before fitting.")

In [ ]:
# --- Variant B: the same information as text ---
if edx is not None:
    for component in m_edx:
        if hasattr(component, "A") and component.active:
            free = "free" if component.A.free else "tied/fixed"
            print(f"{component.name:12s} {float(component.centre.value):7.3f} keV   {free}")

In [ ]:
if edx is not None:
    m_edx.fit_background()
    m_edx.multifit()

In [ ]:
if edx is not None:
    intensities = edx_binned.get_lines_intensity()

    edx_maps = {}
    for signal in intensities:
        # The title reads "X-ray line intensity of : Si_Ka at 1.74 keV", so
        # splitting it is fragile. The metadata carries the line name directly.
        line = signal.metadata.Sample.xray_lines[0]        # e.g. "Si_Ka"
        element = signal.metadata.Sample.elements[0]       # e.g. "Si"
        edx_maps[element] = normalise(np.asarray(signal.data, float))
        print(f"  {element:3s} from {line}")

For the phase rules we need Si, O, S and C - the same four as in EELS. Tungsten
and platinum are not used for the classification itself: sulphur already marks
WS2 and carbon already marks the cap. They are there as an independent check,
which is what exercise 4 uses them for.

In [ ]:
if edx is not None and {"Si", "O", "S", "C"} <= set(edx_maps):
    edx_scores = phase_scores({k: edx_maps[k] for k in ("Si", "O", "S", "C")})
    edx_labels, _ = classify(edx_scores, threshold=THRESHOLD)

    fig, ax = plt.subplots(figsize=(6.5, 5))
    plot_phase_map(edx_labels, phase_names, ax=ax, title="Phase map from EDX")
    plt.show()

    edx_fractions = phase_fractions(edx_labels, phase_names)
elif edx is not None:
    print(f"Need Si, O, S and C. Have: {sorted(edx_maps)}")

## 6. Where the two methods disagree

Two independent measurements of the same thing. Where they agree you can
believe the answer; where they disagree, one of them is wrong and the pattern
of the disagreement usually says which.

Scattered single pixels mean noise. Disagreement along the interfaces means
something physical: the beam samples both phases at once, either because it
broadens on its way through the foil or because the interface is not
perpendicular to it.

In [ ]:
if eels is not None and edx is not None and eels_labels.shape == edx_labels.shape:
    agree = eels_labels == edx_labels
    print(f"identical assignment in {100 * agree.mean():.1f}% of pixels\n")

    print(f"{'phase':14s} {'EELS':>8s} {'EDX':>8s} {'diff':>8s}")
    print("-" * 42)
    for name in ["unassigned"] + phase_names:
        a = eels_fractions.get(name, 0.0) * 100
        b = edx_fractions.get(name, 0.0) * 100
        print(f"{name:14s} {a:7.1f}% {b:7.1f}% {b - a:+7.1f}%")

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
    plot_phase_map(eels_labels, phase_names, ax=axes[0], title="EELS")
    plot_phase_map(edx_labels, phase_names, ax=axes[1], title="EDX")
    axes[2].imshow(agree, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    axes[2].set_title("white = both agree")
    axes[2].set_xticks([]); axes[2].set_yticks([])
    plt.show()
elif eels is not None and edx is not None:
    print(f"Different shapes: EELS {eels_labels.shape}, EDX {edx_labels.shape}.")
    print("Use the same BINNING for both, or rebin one of them to match.")

## 7. Bonus: telling Si from SiO2 without using oxygen

Everything above separates Si from SiO2 through the oxygen map. There is a
second, independent route: the **shape** of the silicon L2,3 edge itself.

Elemental silicon and silicon dioxide are the same element in different
chemical surroundings, and the fine structure in the first 30 eV above the edge
reflects that. In SiO2 the edge onset shifts up by roughly 6 eV and grows a
sharp white line; in elemental Si it stays low and rounded. So the ELNES is a
fingerprint, and a linear combination of the two measured references tells you
the mixture in every pixel - without oxygen entering the argument at all.

**This section runs with the workshop's own reference spectra**
(`data/nanopore/Si Standards/`, which contain `Si.msa` and `SiO2.msa`), so you
can try the method before your own data arrives.

Why it is worth having a second route: at a thin interface the oxygen map is
blurred by beam broadening, while the fine structure comes from the same pixel
as the silicon signal. The two disagree in interesting places.

In [ ]:
from workshop_data import load_standards
from scipy.ndimage import gaussian_filter1d

# --- EDIT: which references, and which window to fit ---
STANDARDS_FOLDER = "nanopore/Si Standards"
FIT_WINDOW       = (95.0, 140.0)   # across the Si-L2,3 edge and its fine structure
SMOOTHING        = 2
# -------------------------------------------------------

# This part works with the workshop's own references, with or without your data.
standards = load_standards(folder=STANDARDS_FOLDER, sigma=SMOOTHING)
keep = [n for n in standards if n.lower() in ("si", "sio2")]
print("using references:", keep)

fig, ax = plt.subplots(figsize=(7, 4))
for name in keep:
    s = standards[name]
    axis = s.axes_manager[-1].axis
    ax.plot(axis, s.data / s.data.max(), lw=1.4, label=name)
ax.axvspan(*FIT_WINDOW, color="0.85", zorder=0, label="fit window")
ax.set_xlabel("energy loss / eV")
ax.set_ylabel("normalised intensity")
ax.set_title("Si-L2,3 fine structure: elemental Si vs SiO2")
ax.legend()
plt.show()

The two curves above are the whole idea. If they look identical, the fit that
follows cannot work and something is wrong with the references.

Now the same preprocessing for references and data - otherwise you fit
background-carrying curves against background-free ones - and then a fit whose
only free parameters are the two mixing weights.

In [ ]:
if eels is not None:
    binned = eels.rebin(scale=BINNING)
    binned = binned.remove_background(signal_range=(80.0, 95.0))
    binned = binned.isig[FIT_WINDOW[0]:FIT_WINDOW[1]]

    elnes = binned.deepcopy()
    elnes.data = gaussian_filter1d(elnes.data, sigma=SMOOTHING, axis=-1)

    references = {}
    for name in keep:
        s = standards[name].deepcopy()
        s.set_signal_type("EELS")
        s = s.isig[FIT_WINDOW[0]:FIT_WINDOW[1]]
        s.data = s.data / s.data.max()
        references[name] = s
        print(f"{name:8s} {s.axes_manager[-1].size} channels")

In [ ]:
if eels is not None:
    # auto_add_edges=False: the two references already describe the edge.
    # Adding a physical Si-L edge on top would model the same feature twice
    # and the fit would have no unique answer.
    m_elnes = elnes.create_model(auto_background=False, auto_add_edges=False)

    for name, s in references.items():
        pattern = hs.model.components1D.ScalableFixedPattern(s)
        pattern.name = name
        pattern.xscale.free = False
        pattern.shift.free = False
        pattern.yscale.bmin = 0      # a negative amount of a phase is meaningless
        pattern.yscale.bmax = 1e7
        m_elnes.append(pattern)

    m_elnes.components

In [ ]:
# --- Variant A: interactive ---
#
# CAUTION: close this widget again before fitting.
#
# m.gui() binds every model parameter to a slider. While a fit runs, the optimiser
# writes each parameter thousands of times per second; every value travels to the
# browser and comes back with float32 precision, and the binding then stops the fit
# with
#
#   TraitError: Broken link ...: the source value changed while updating the target
#
# If that happens to you, re-run the cell above that creates the model - a fresh
# model carries no widget - and fit again. No kernel restart needed.

SHOW_GUI = False

if SHOW_GUI and eels is not None:
    m_elnes.gui()
else:
    print("interactive widget off - set SHOW_GUI = True to open it,")
    print("and re-create the model before fitting.")

In [ ]:
# --- Variant B: the same information as text ---
if eels is not None:
    for component in m_elnes:
        yscale = getattr(component, "yscale", None)
        if yscale is None:
            print(f"{component.name:20s} (no yscale)")
        else:
            print(f"{component.name:20s} yscale={yscale.value}")

In [ ]:
if eels is not None:
    m_elnes.multifit(bounded=True)

In [ ]:
if eels is not None:
    weights = {c.name: np.asarray(c.yscale.map["values"], float) for c in m_elnes}
    total = sum(weights.values())
    with np.errstate(invalid="ignore", divide="ignore"):
        oxide_fraction = np.where(total > 0, weights.get("SiO2", 0) / total, np.nan)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    for ax, (name, w) in zip(axes, weights.items()):
        ax.imshow(w, cmap="magma")
        ax.set_title(f"{name} weight"); ax.set_xticks([]); ax.set_yticks([])
    im = axes[2].imshow(oxide_fraction, cmap="coolwarm", vmin=0, vmax=1)
    axes[2].set_title("SiO2 / (Si + SiO2)")
    axes[2].set_xticks([]); axes[2].set_yticks([])
    fig.colorbar(im, ax=axes[2], shrink=0.8)
    plt.show()

    print("0 = elemental Si, 1 = fully oxidised, in between = mixed pixel")

## Exercises

Roughly in order of how much they teach.

**1. Predict before you compute.**
Before running section 2, write down for each of the four phases which EELS
edge and which EDX line you would use. Then run it. Which of your choices does
eXSpy say are impossible, and would you have noticed if the check had not told
you?

**2. Walk into the silicon trap on purpose.**
Pick a region you know is WS2. Fit the EDX spectrum there with `["Si", "S"]`
only - no tungsten in the element list. How much silicon does it report? Now
add `"W"` and refit. The ratio between those two numbers is the most important
figure in this notebook: it is the error you would have published.

**3. Choose the tungsten line yourself.**
Quantify W once with `W_Ma` and once with `W_La` (force them with
`add_lines(["W_Ma", ...])`). Compare the maps. One is noisier, the other is
wrong. Explain in one sentence why you take the noisier one - and what you
would change in the acquisition to make it less noisy.

**4. Let stoichiometry mark your work.**
In the WS2 region, form the S:W intensity ratio from EDX. You already know the
answer is 2:1. How far off are you? Name the three effects that stand between a
raw intensity ratio and a real composition, and say which one you think
dominates here.

**5. Carbon: easy for EELS, hard for EDX.**
Compare the carbon map from EELS with the one from EDX. The EDX one will be
much worse. Explain it with the fluorescence yield at Z = 6 and with absorption
on the way to the detector. What does that imply for mapping any light element
next to heavy ones?

**6. How much does the threshold decide?**
Vary `THRESHOLD` from 0.05 to 0.4 and watch the phase boundaries move. Which
interface moves most? A boundary that shifts a lot under a change of threshold
is a boundary your data does not actually resolve - so which of your interfaces
are real, and which are wishful thinking?

**7. Read the disagreement.**
Take the "both agree" image from section 6. Are the disagreeing pixels
scattered at random, or lined up along interfaces? Scattered means noise; lined
up means the beam is seeing two phases at once. Estimate how thick an interface
would have to be for you to resolve it, given your pixel size and the foil
thickness.

**8. Bonus: two routes to the same boundary.**
Compare the `SiO2/(Si+SiO2)` map from section 7 with the Si/SiO2 boundary from
section 4, which used oxygen. Where do they disagree? At a thin interface, which
of the two do you trust more, and why? (Think about which signal comes from the
same pixel as the silicon, and which one has been smeared.)

**9. Design the next acquisition.**
Given everything above, write down three concrete changes for the next session:
energy window, dispersion, dwell time, beam current, detector, binning. For each
one, name the specific result in this notebook that made you want it.

## A note on your energy window

You said 80-600 eV. That covers Si-L2,3 (99), S-L2,3 (165), S-L1 (229),
C-K (284) and O-K (532), so all four phases are reachable - but it leaves only
**68 eV above the oxygen edge**.

That is enough to map oxygen. It is thin for anything quantitative: the
integration window behind O-K is short, and the power-law background in front
of it has to be extrapolated a long way from the last clean stretch. If your
spectrometer reaches 650-700 eV at the same dispersion, take it - it costs
nothing and makes the oxygen numbers much steadier.

If you keep 600 eV, treat the oxygen map as qualitative. The Si/SiO2 separation
in section 7 does not depend on oxygen at all, which is a good reason to have it.